# テキストデータを人、課題、解決策に変換する

### 1. インスタンスの生成

In [1]:
from langchain_openai import ChatOpenAI
from utils.langchain import Langchain
from utils.file_reader import FileReader
import os

llm = ChatOpenAI(
    model="gpt-5",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    max_tokens=None,
    timeout=None,
    max_retries=3
)

langchain = Langchain(llm)
file_reader = FileReader()

### 2. データセットの読み込み

In [2]:
dataset_dir = "/workspace/datasets/data/preprocessed_texts/insurance_v2"

# List all paths in the dataset directory
dataset_names = [
    f for f in os.listdir(dataset_dir)
]

print("Available datasets:", dataset_names, "\n" , len(dataset_names))

Available datasets: ['IR_100128374150231217.txt', 'IR_200106816650240531.txt', 'IR_200118630950240629.txt', 'IR_200157197950240930.txt', 'IR_100115764450231123.txt', 'IR_100132118050231225.txt', 'IR_200108439850240604.txt', 'IR_100196502150240505.txt', 'IR_200105430150240528.txt', 'IR_100128401350231217.txt', 'IR_200107292450240601.txt', 'IR_200116013450240623.txt', 'IR_100194726450240430.txt', 'IR_100198790950240511.txt', 'IR_200160746450241009.txt', 'IR_100122570450231206.txt', 'IR_100128732550231218.txt', 'IR_100195218150240501.txt', 'IR_200115891350240622.txt', 'IR_100131683150231224.txt', 'IR_100116263350231124.txt', 'IR_200150129250240912.txt', 'IR_200118205450240628.txt', 'IR_100126912650231214.txt', 'IR_200105459250240528.txt', 'IR_200160869250241009.txt', 'IR_100121619050231204.txt', 'IR_100112876650231117.txt', 'IR_200113223250240615.txt', 'IR_200151020550240914.txt', 'IR_100111647250231115.txt', 'IR_100111471850231115.txt', 'IR_100107748150231109.txt', 'IR_100122727550231206

### 3. タプルの生成

In [ ]:
import os
import json
from pathlib import Path
from prompt_v4 import PROMPT_STEP1, PROMPT_STEP2
from langchain_openai import ChatOpenAI
from utils.langchain import Langchain
from utils.file_reader import FileReader
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Config ---
dataset_dir = "/workspace/datasets/data/preprocessed_texts/insurance_v2"
output_dir_1 = "/workspace/outputs_sample/insurance_v2/"

os.makedirs(output_dir_1, exist_ok=True)

# --- LLM / helpers ---
llm = ChatOpenAI(
    model="gpt-5",
    api_key=os.getenv("OPENAI_API_KEY"),
    max_tokens=None,
    timeout=None,
    max_retries=3,
)

langchain = Langchain(llm)
file_reader = FileReader()

def clean_response(content: str) -> str:
    if content.startswith("```"):
        content = content[3:]
    if content.startswith("json"):
        content = content[4:]
    if content.endswith("```"):
        content = content[:-3]
    return content.strip()

def process(fname: str) -> tuple[str, bool]:
    file_id = Path(fname).stem
    src_path = os.path.join(dataset_dir, fname)
    conversation = file_reader.read_file(src_path)
    
    step1_prompt = PROMPT_STEP1.replace("$conversation", conversation)
    res1 = langchain.generate_prompt_only(step1_prompt)
    content1 = getattr(res1, "content", "") if res1 is not None else ""
    content1 = clean_response(content1)

    out1_path = os.path.join(output_dir_1, f"{file_id}.json")
    with open(out1_path, "w", encoding="utf-8") as f:
        f.write(content1)

    print(f"Wrote {out1_path}")
    return file_id, True

# --- Process files ---
txt_files = [f for f in os.listdir(dataset_dir) if f.endswith(".txt")]
txt_files = sorted(txt_files)

with ThreadPoolExecutor() as executor:
    futures = {executor.submit(process, fname): fname for fname in txt_files}
    for future in as_completed(futures):
        fname = futures[future]
        try:
            future.result()
        except Exception as e:
            print(f"⚠️ Error processing {fname}: {e}")

print("Done.")

[Step1] Wrote /workspace/20251122_test/outputs/insurance_v2/step_1/IR_100107748150231109.json
[Step1] Wrote /workspace/20251122_test/outputs/insurance_v2/step_1/IR_100108299150231110.json
[Step1] Wrote /workspace/20251122_test/outputs/insurance_v2/step_1/IR_100110093150231113.json
[Step1] Wrote /workspace/20251122_test/outputs/insurance_v2/step_1/IR_100111471850231115.json
[Step1] Wrote /workspace/20251122_test/outputs/insurance_v2/step_1/IR_100111706250231115.json
[Step1] Wrote /workspace/20251122_test/outputs/insurance_v2/step_1/IR_100108755650231111.json
[Step1] Wrote /workspace/20251122_test/outputs/insurance_v2/step_1/IR_100112149950231116.json
[Step1] Wrote /workspace/20251122_test/outputs/insurance_v2/step_1/IR_100109782650231112.json
[Step1] Wrote /workspace/20251122_test/outputs/insurance_v2/step_1/IR_100111647250231115.json
[Step1] Wrote /workspace/20251122_test/outputs/insurance_v2/step_1/IR_100112876650231117.json
[Step1] Wrote /workspace/20251122_test/outputs/insurance_v2/